In [1]:
#   6.1 制作数据集

#   这一章我们需要在torchvision库中分别下载训练集与测试集，因此需要从
# torchvision库中导入datasets以下载数据集，下载前需要借助torchvision库中
# 的transforms进行图像转换，将数据集变为张量，并调整数据集的统计分布。
#   由于不需要手动构建数据集，因此不导入utils中的Dataset;又由于训练集
# 与测试集是分开下载的，因此不导入utils中的random_split。

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
%matplotlib inline

# 展示高清图
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')

# 在下载数据集之前，要设定转换参数：transform,该参数里解决两个问题：
# ● ToTensor:将图像数据转为张量，且调整三个维度的顺序为C*W*H;
# C表示通道数，二维灰度图像的通道数为1，三维RGB彩图的通道数为3。
# ● Normalize:将神经网络的输入数据转化为标准正态分布，训练更好；
# 根据统计计算，MNIST训练集所有像素的均值是0.1307、标准差是0.3081。

# 设定下载参数
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.1307, 0.3081)
])

# 下载训练集与测试集
train_Data = datasets.MNIST(
    root = 'E:\Deep_Learning\DNN',
    train = True,       # 表示下载的是训练集
    download = True,    # 如果在文件夹中没有检索到，则进行下载
    transform = transform
)

test_Data = datasets.MNIST(
    root = 'E:\Jupyter_DL\Deep_Learning\DNN',
    train = False,       # 表示下载的是测试集
    download = True,     # 如果在文件夹中没有检索到，则进行下载
    transform = transform
)

# 批次加载器
train_loader = DataLoader(dataset=train_Data, batch_size=128, shuffle=True)     # shuffle用于在每个epoch内先洗牌再分批
test_loader = DataLoader(dataset=test_Data, batch_size=64, shuffle=False)

<>:34: SyntaxWarning: invalid escape sequence '\D'
<>:41: SyntaxWarning: invalid escape sequence '\J'
<>:34: SyntaxWarning: invalid escape sequence '\D'
<>:41: SyntaxWarning: invalid escape sequence '\J'
C:\Users\31726\AppData\Local\Temp\ipykernel_13296\2456652616.py:34: SyntaxWarning: invalid escape sequence '\D'
  root = 'E:\Deep_Learning\DNN',
C:\Users\31726\AppData\Local\Temp\ipykernel_13296\2456652616.py:41: SyntaxWarning: invalid escape sequence '\J'
  root = 'E:\Jupyter_DL\Deep_Learning\DNN',
100%|██████████| 9.91M/9.91M [00:20<00:00, 476kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 96.8kB/s]
100%|██████████| 1.65M/1.65M [00:03<00:00, 448kB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]
100%|██████████| 9.91M/9.91M [00:21<00:00, 472kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 96.6kB/s]
100%|██████████| 1.65M/1.65M [00:03<00:00, 470kB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [2]:
#   6.2 搭建神经网络

# 每个样本的输入都是形状为28×28的二维数组，那么对于DNN来说，
# 输入层的神经元节点就要有28×28=784个；输出层使用独热编码，需要10个节点(数字0~9)

class DNN(nn.Module):
    def __init__(self):
        '''搭建神经网络各层'''
        super(DNN, self).__init__()
        self.net = nn.Sequential(           # 按顺序搭建各层
            nn.Flatten(),                   # 先把图像铺平成一维
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10)
        )

def forward(self, x):
        '''前向传播'''
        y = self.net(x)     # x即输入数据
        return y            # y即输出数据

model = DNN()       # 创建子类的实例
# model = DNN().to('cuda:0')    # 搬到GPU上
print(model)        # 查看实例的各层

DNN(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Linear(in_features=256, out_features=128, bias=True)
    (6): ReLU()
    (7): Linear(in_features=128, out_features=64, bias=True)
    (8): ReLU()
    (9): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [3]:
#   6.3 训练网络

# 损失函数的选择（交叉熵损失）
loss_fn = nn.CrossEntropyLoss() # 自带softmax激活函数

# 优化算法的选择
learning_rate = 0.01   # 设置学习率
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.5
# 给优化器了一个新参数momentum(动量)，它使梯度下降算法
# 有了力与惯性，该方法给人的感觉就像是小球在地面上滚动一样。
)

epochs = 5
losses = []     # 记录损失函数变化的列表

for epoch in range(epochs):
    for (x, y) in train_loader: # 获取小批次的x与y
#        x, y = x.to('cuda:0'), y.to('cuda:0')  #  把小批次搬到GPU上
        Pred = model(x)             # 一次前向传播（BGD）
        loss = loss_fn(Pred, y)     # 计算损失函数
        losses.append(loss.item())  # 记录损失函数的变化
        optimizer.zero_grad()       # 清理上一轮滞留的梯度
        loss.backward()             # 一次反向传播
        optimizer.step()            # 优化内部参数

Fig = plt.figure()
plt.plot(range(len(losses)), losses)
plt.show()

NotImplementedError: Module [DNN] is missing the required "forward" function

In [ ]:
#   6.4 测试网络

correct = 0
total = 0

with torch.no_grad():       # 关闭梯度计算功能
    for x, y in test_loader:    # 获取小批次的x与y
#        x, y = x.to('cuda:0'), y.to('cuda:0')  #  把小批次搬到GPU上
        Pred = model(x)          # 一次前向传播(小批量)
#   a,b=torch.max(Pred.data,dim=1)的意思是，找出Pred每一行里的最大值，
# 数值赋给a,所处位置赋给b。因此上述代码里的pred就相当于把独热编码
# 转换回了普通的阿拉伯数字，这样一来可以直接与y进行比较。
#   由于此处pred与y是一阶张量，因此correct行的结尾不能加.all(1)e]
        _, pred = torch.max(Pred.data, dim=1)
        correct += torch.sum( (pred == y) )   # 预测正确的样本
        total += y.size(0)                       # 全部的样本数量

print(f'测试集精准度:{100*correct/total} % ')